# RailSafe — Train on Kaggle / Colab (1-Click)
Clone repo, install deps, download 2/3 primary datasets, build manifest, train YOLO.
- **Kaggle:** New Notebook → Add Data → GitHub → `YOUR/RAILSAFE`
- **Colab:** `!git clone https://github.com/YOUR/RAILSAFE.git`
RFDD 8.24GB gated — training works without it (surface_faults 5,153 + railsense 858).


In [ ]:
# 0. Clone (Colab only — Kaggle already has repo via Add Data)
# !git clone https://github.com/YOUR/RAILSAFE.git
# %cd RAILSAFE
!ls -la

In [ ]:
# 1. Setup: install torch+ultralytics, download railsense+surface, build manifest
# On Kaggle, add Kaggle API token via Secrets: KAGGLE_USERNAME + KAGGLE_KEY (optional, for railsense mirror)
!python scripts/setup_kaggle.py --datasets railsense surface_faults
!cat datasets/manifest.jsonl | head -n 3
!ls -lh datasets/yolo_surface/train 2>&1 | head

In [ ]:
# 2. Dry run check (no GPU needed)
!python scripts/train.py --task yolo --dry-run

In [ ]:
# 3. Train YOLO11n (2.6M) for 5 epochs — ~10 min on Kaggle T4, 1.5GB VRAM
# Use yolo11m.pt (20.1M) for best RFDD 0.7106 mAP50:95 (needs RFDD)
!python scripts/train.py --task yolo --epochs 5 --model yolo11n.pt
!ls -lh runs/yolo/surface_classify* 2>&1 | tail

In [ ]:
# 4. (Optional) Phase 1 RailSense — needs TF 2.16, ~30 min on CPU
# !pip install -q tensorflow==2.16.1
# !python scripts/train.py --task railsense --epochs 5

In [ ]:
# 5. Evaluate severity/risk engines (no training)
!python -c "from ml.severity.severity_engine import compute_severity; print(compute_severity(0.81,'Cracks',0.27,'rail'))"
!python -c "from ml.risk.risk_engine import compute_risk_v1; print(compute_risk_v1(0.68,0.85,0.27))"